
> **Public reproducibility notebook.** This notebook contains the experiment logic used in the paper. Machine-specific paths have been replaced with repository-relative configuration, and saved outputs are intentionally cleared.


# External Retrieval Baseline: SARAF Retrieval Rule under a Matched Protocol

This notebook is a **final external-baseline experiment** for the ICLR submission.

## Why this experiment?

Our main controlled experiments compare:

- Pattern
- Candidate Prior
- Future-Compatible Learned
- Shuffled Future

Those controls identify *why* historical examples are relevant, but a reviewer may also ask whether the learned retriever is competitive with a recent **external retrieval method**.

We therefore add SARAF (KDD 2026) as the closest external retrieval principle.

---

## Important scope

This notebook evaluates the **SARAF retrieval rule under our frozen matched protocol**:

\[
L=96,\quad H\in\{24,48,96\},\quad M=100,\quad K=10
\]

with:

- same-channel candidate pool,
- fixed train-scale future target,
- exactly the same Pattern Top-\(M\) candidates used by our confirmatory experiment,
- the same test queries,
- the same AnalogFutureMSE evaluation.

This is **not** the authors' native forecasting reproduction.

The official SARAF scripts use a different forecasting protocol (notably `seq_len=720` and dataset/horizon-specific retrieval counts). Therefore, in the paper this result should be called:

> **SARAF retrieval rule (matched protocol)**

or

> **SARAF-MMR (matched retrieval protocol)**

unless an exact native reproduction is additionally performed.

---

## SARAF components reproduced here

The official public implementation:

1. estimates a dataset-level stationarity score from changes in rolling mean and variance;
2. maps stationarity \(s\in[0,1]\) to

\[
\lambda(s)=0.3+0.6s
\]

for MMR relevance/diversity balance;

3. first forms a Top-100 similarity candidate pool;
4. chooses the most similar candidate first;
5. subsequently performs stochastic MMR:

\[
\mathrm{MMR}(i)
=
\lambda s_i
-
(1-\lambda)\max_{j\in S}\mathrm{sim}(i,j)
\]

and samples according to

\[
p(i)\propto
\exp(\mathrm{MMR}(i)/0.1).
\]

The public code approximates candidate-to-candidate redundancy from the closeness of their query-similarity scores:

\[
\mathrm{sim}(i,j)
=
1-|s_i-s_j|.
\]

We reproduce these rules directly.

---

## Datasets

The matched comparison is run on the **four final confirmatory datasets**:

- Electricity
- Traffic
- Exchange
- Solar

These are sufficient for the external-baseline check because the same frozen five-seed confirmatory outputs already exist for our Learned retriever.

> **Public repository version.** Paths are repository-relative by default.
> Set `WHM_DATA_ROOT` to use datasets stored elsewhere and `WHM_WORK_ROOT` to move generated caches/checkpoints outside the repository.
> Saved execution outputs were cleared intentionally so the notebook does not expose machine-specific paths or stale results.


In [ ]:
from pathlib import Path
import os

def _find_repo_root(start=None):
    """Locate the repository root from the current working directory."""
    start = Path(start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "experiments").exists():
            return candidate
    raise RuntimeError(
        "Repository root not found. Start Jupyter from inside the cloned "
        "which-histories-matter repository, or set WHM_REPO_ROOT."
    )

_env_repo = os.environ.get("WHM_REPO_ROOT")
REPO_ROOT = Path(_env_repo).expanduser().resolve() if _env_repo else _find_repo_root()
REPO_DATA_ROOT = Path(os.environ.get("WHM_DATA_ROOT", REPO_ROOT / "data")).expanduser().resolve()
REPO_WORK_ROOT = Path(os.environ.get("WHM_WORK_ROOT", REPO_ROOT / "_work")).expanduser().resolve()
REPO_WORK_ROOT.mkdir(parents=True, exist_ok=True)

print("Repository root:", REPO_ROOT)
print("Data root:", REPO_DATA_ROOT)
print("Work root:", REPO_WORK_ROOT)


## 0. Configuration

In [ ]:

from pathlib import Path
import math
import random
import warnings

import numpy as np
import pandas as pd

import torch
import torch.nn.functional as F

warnings.filterwarnings("ignore")

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

CONFIRM_DIR = REPO_WORK_ROOT / "final_confirmatory"

CACHE_DIR = CONFIRM_DIR / "cache"

DATA_ROOT = REPO_DATA_ROOT

DATA_PATHS = {
    "Electricity":
        DATA_ROOT / "electricity/electricity.csv",

    "Traffic":
        DATA_ROOT / "traffic/traffic.csv",

    "Exchange":
        DATA_ROOT / "exchange_rate/exchange_rate.csv",

    "Solar":
        DATA_ROOT / "Solar/solar_AL.txt",
}

DATASETS = [
    "Electricity",
    "Traffic",
    "Exchange",
    "Solar",
]

HORIZONS = [24, 48, 96]

SEQ_LEN = 96
TOP_M = 100
TOP_K = 10

# Official SARAF Retrieval.py defaults/ranges.
SIGMA_MIN = 0.05
SIGMA_MAX = 0.30
LAMBDA_MIN = 0.30
LAMBDA_MAX = 0.90
MMR_TEMPERATURE = 0.10

# Five stochastic retrieval seeds for a fair stability check.
SARAF_SEEDS = [0, 1, 2, 3, 4]

BLOCK_ANCHORS = 10
N_BOOT = 5000

RESULT_DIR = (
    CONFIRM_DIR /
    "external_baselines" /
    "saraf_matched"
)

RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("Device:", DEVICE)
print("Confirmatory directory:", CONFIRM_DIR)
print("Output directory:", RESULT_DIR)

assert CONFIRM_DIR.exists(), (
    "Run the final confirmatory notebook first."
)


## 1. Load the exact frozen channel selection from the confirmatory experiment

In [ ]:

channel_selection_path = (
    CONFIRM_DIR /
    "01_channel_selection.csv"
)

assert channel_selection_path.exists()

channel_selection_summary = pd.read_csv(
    channel_selection_path
)

display(
    channel_selection_summary
)


In [ ]:

def load_standard_csv(path):
    df = pd.read_csv(path)

    timestamp_cols = [
        c for c in df.columns
        if str(c).lower() in {
            "date",
            "datetime",
            "timestamp",
            "time",
        }
    ]

    x = (
        df
        .drop(
            columns=timestamp_cols,
            errors="ignore",
        )
        .apply(
            pd.to_numeric,
            errors="coerce",
        )
    )

    good_cols = [
        c for c in x.columns
        if x[c].notna().mean() > 0.99
    ]

    x = x[good_cols]

    x = (
        x
        .replace(
            [np.inf, -np.inf],
            np.nan,
        )
        .interpolate(
            axis=0,
            limit_direction="both",
        )
        .ffill()
        .bfill()
    )

    return x


def load_solar_txt(path):
    x = pd.read_csv(
        path,
        header=None,
    )

    if x.shape[1] == 1:
        x = pd.read_csv(
            path,
            header=None,
            sep=r"\s+",
        )

    x = (
        x
        .apply(
            pd.to_numeric,
            errors="coerce",
        )
        .replace(
            [np.inf, -np.inf],
            np.nan,
        )
        .interpolate(
            axis=0,
            limit_direction="both",
        )
        .ffill()
        .bfill()
    )

    x.columns = [
        f"Solar_{i:03d}"
        for i in range(x.shape[1])
    ]

    return x


RAW = {}

for dataset_name in DATASETS:
    path = DATA_PATHS[dataset_name]

    assert path.exists(), (
        dataset_name,
        path,
    )

    if dataset_name == "Solar":
        RAW[dataset_name] = load_solar_txt(
            path
        )
    else:
        RAW[dataset_name] = load_standard_csv(
            path
        )

    print(
        dataset_name,
        RAW[dataset_name].shape,
    )


In [ ]:

# Reproduce the deterministic channel-selection rule used in the
# final confirmatory notebook.

MAX_CHANNELS = {
    "Electricity": 32,
    "Traffic": 32,
    "Exchange": None,
    "Solar": None,
}

SELECTED_CHANNELS = {}
CHANNEL_NORMALIZED = {}
SPLITS = {}

for dataset_name in DATASETS:

    df = RAW[dataset_name]

    n = len(df)

    train_end = int(0.70 * n)
    val_end = int(0.80 * n)

    SPLITS[dataset_name] = {
        "train_end": train_end,
        "val_end": val_end,
    }

    train = df.iloc[:train_end]

    train_std = train.std(
        axis=0,
        ddof=0,
    )

    valid_cols = [
        c for c in df.columns
        if (
            np.isfinite(train_std[c])
            and
            train_std[c] > 1e-6
        )
    ]

    max_c = MAX_CHANNELS[dataset_name]

    if (
        max_c is not None
        and len(valid_cols) > max_c
    ):
        idx = np.linspace(
            0,
            len(valid_cols) - 1,
            max_c,
            dtype=int,
        )

        selected = [
            valid_cols[i]
            for i in idx
        ]
    else:
        selected = valid_cols

    SELECTED_CHANNELS[dataset_name] = selected

    df_sel = df[selected]

    mu = (
        df_sel
        .iloc[:train_end]
        .mean(axis=0)
        .to_numpy(
            dtype=np.float32
        )
    )

    sd = (
        df_sel
        .iloc[:train_end]
        .std(
            axis=0,
            ddof=0,
        )
        .to_numpy(
            dtype=np.float32
        )
    )

    arr = df_sel.to_numpy(
        dtype=np.float32
    )

    z = (
        arr -
        mu[None, :]
    ) / sd[None, :]

    CHANNEL_NORMALIZED[
        dataset_name
    ] = z.astype(
        np.float32
    )

    print(
        dataset_name,
        "| channels:",
        len(selected),
        "| train_end:",
        train_end,
    )



## 2. SARAF dataset-level stationarity

The official SARAF implementation divides each past window into six temporal blocks and measures variation of:

- block means,
- block standard deviations,

relative to the global window standard deviation.

Larger values indicate greater stationarity.

For the matched protocol, we apply the same statistic to the train-normalized multivariate windows over the selected channels.


In [ ]:

@torch.no_grad()
def saraf_stationarity_batch(
    x,
):
    # x: [B, L, C]
    B, L, C = x.shape

    n_windows = 6
    window_size = L // n_windows

    window_means = []
    window_stds = []

    for i in range(n_windows):

        start = i * window_size

        end = (
            start + window_size
            if i < n_windows - 1
            else L
        )

        w = x[:, start:end, :]

        window_means.append(
            w.mean(dim=1)
        )

        window_stds.append(
            w.std(
                dim=1,
                unbiased=True,
            )
        )

    window_means = torch.stack(
        window_means,
        dim=1,
    )

    window_stds = torch.stack(
        window_stds,
        dim=1,
    )

    mean_variation = (
        window_means
        .std(
            dim=1,
            unbiased=True,
        )
        .mean(dim=1)
    )

    std_variation = (
        window_stds
        .std(
            dim=1,
            unbiased=True,
        )
        .mean(dim=1)
    )

    global_std = (
        x.std(
            dim=1,
            unbiased=True,
        )
        .mean(dim=1)
    )

    mean_score = 1.0 - (
        mean_variation /
        (
            global_std +
            1e-8
        )
    ).clamp(
        0,
        1,
    )

    std_score = 1.0 - (
        std_variation /
        (
            global_std +
            1e-8
        )
    ).clamp(
        0,
        1,
    )

    stationarity = (
        0.5 *
        mean_score
        +
        0.5 *
        std_score
    ).clamp(
        0,
        1,
    )

    return stationarity


def compute_dataset_stationarity(
    dataset_name,
    batch_size=1024,
):
    # Use all temporally valid training windows, matching SARAF's
    # dataset-level averaging idea. This is the selected-channel
    # matched protocol rather than native seq_len=720 SARAF.
    arr = CHANNEL_NORMALIZED[
        dataset_name
    ]

    train_end = SPLITS[
        dataset_name
    ][
        "train_end"
    ]

    anchors = np.arange(
        SEQ_LEN,
        train_end,
        dtype=np.int64,
    )

    total = 0.0
    count = 0

    for start in range(
        0,
        len(anchors),
        batch_size,
    ):

        a = anchors[
            start:
            start +
            batch_size
        ]

        windows = np.stack(
            [
                arr[
                    t -
                    SEQ_LEN:
                    t
                ]
                for t in a
            ],
            axis=0,
        )

        x = torch.tensor(
            windows,
            dtype=torch.float32,
            device=DEVICE,
        )

        s = saraf_stationarity_batch(
            x
        )

        total += float(
            s.sum().item()
        )

        count += int(
            len(s)
        )

    return total / count


stationarity_rows = []

SARAF_PARAMS = {}

for dataset_name in DATASETS:

    s = compute_dataset_stationarity(
        dataset_name
    )

    sigma = (
        SIGMA_MIN
        +
        (1.0 - s) *
        (
            SIGMA_MAX -
            SIGMA_MIN
        )
    )

    lambda_val = (
        LAMBDA_MIN
        +
        s *
        (
            LAMBDA_MAX -
            LAMBDA_MIN
        )
    )

    SARAF_PARAMS[
        dataset_name
    ] = {
        "stationarity":
            s,

        "sigma":
            sigma,

        "lambda":
            lambda_val,
    }

    stationarity_rows.append({
        "Dataset":
            dataset_name,

        "Stationarity":
            s,

        "AdaptiveSigma":
            sigma,

        "AdaptiveLambda":
            lambda_val,
    })

stationarity_table = pd.DataFrame(
    stationarity_rows
)

display(
    stationarity_table
)

stationarity_table.to_csv(
    RESULT_DIR /
    "saraf_stationarity_parameters.csv",
    index=False,
)


## 3. Load the exact Pattern Top-100 pools and test futures

In [ ]:

def load_task(
    dataset_name,
    H,
):
    win_path = (
        CACHE_DIR /
        f"{dataset_name}_H{H}_windows.npz"
    )

    meta_path = (
        CACHE_DIR /
        f"{dataset_name}_H{H}_meta.csv.gz"
    )

    topm_path = (
        CACHE_DIR /
        f"{dataset_name}_H{H}_same_topM.npz"
    )

    assert win_path.exists()
    assert meta_path.exists()
    assert topm_path.exists()

    w = np.load(
        win_path
    )

    meta = pd.read_csv(
        meta_path
    )

    topm = np.load(
        topm_path
    )

    q_idx = topm[
        "test_query"
    ].astype(
        np.int64
    )

    cand_idx = topm[
        "test_idx"
    ].astype(
        np.int64
    )

    pattern_score = topm[
        "test_score"
    ].astype(
        np.float32
    )

    future = w[
        "future"
    ].astype(
        np.float32
    )

    q_future = future[
        q_idx
    ]

    cand_future = future[
        cand_idx
    ]

    q_anchor = (
        meta
        .iloc[q_idx][
            "Anchor"
        ]
        .to_numpy(
            dtype=np.int64
        )
    )

    q_channel = (
        meta
        .iloc[q_idx][
            "ChannelIndex"
        ]
        .to_numpy(
            dtype=np.int64
        )
    )

    return {
        "q_idx":
            q_idx,

        "cand_idx":
            cand_idx,

        "pattern_score":
            pattern_score,

        "q_future":
            q_future,

        "cand_future":
            cand_future,

        "q_anchor":
            q_anchor,

        "q_channel":
            q_channel,
    }


for dataset_name in DATASETS:
    for H in HORIZONS:
        x = load_task(
            dataset_name,
            H,
        )

        assert x[
            "pattern_score"
        ].shape[1] == TOP_M

        print(
            dataset_name,
            H,
            "| test queries:",
            len(
                x[
                    "q_idx"
                ]
            ),
        )



## 4. Batched stochastic MMR

This follows the public SARAF retrieval code:

- first candidate: highest query similarity;
- redundancy proxy: \(1-|s_i-s_j|\);
- adaptive MMR balance \(\lambda(s)\);
- stochastic sampling with temperature 0.1.


In [ ]:

def set_seed(
    seed,
):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


@torch.no_grad()
def saraf_mmr_batched(
    pattern_scores,
    lambda_val,
    k=TOP_K,
    temperature=MMR_TEMPERATURE,
    batch_size=512,
    seed=0,
):
    # pattern_scores: [N, M], descending Pattern Top-M scores.
    set_seed(seed)

    all_selected = []

    for start in range(
        0,
        len(pattern_scores),
        batch_size,
    ):

        s_np = pattern_scores[
            start:
            start +
            batch_size
        ]

        score = torch.tensor(
            s_np,
            dtype=torch.float32,
            device=DEVICE,
        )

        B, M = score.shape

        # Official SARAF redundancy proxy:
        # 1 - abs(candidate query-similarity differences)
        cand_sim_matrix = (
            1.0
            -
            torch.abs(
                score[:, :, None]
                -
                score[:, None, :]
            )
        )

        selected = torch.empty(
            (
                B,
                k,
            ),
            dtype=torch.long,
            device=DEVICE,
        )

        selected_mask = torch.zeros(
            (
                B,
                M,
            ),
            dtype=torch.bool,
            device=DEVICE,
        )

        first = torch.argmax(
            score,
            dim=1,
        )

        selected[:, 0] = first

        selected_mask.scatter_(
            1,
            first[:, None],
            True,
        )

        row = torch.arange(
            B,
            device=DEVICE,
        )

        max_sim_to_selected = (
            cand_sim_matrix[
                row,
                :,
                first
            ]
        )

        for step in range(
            1,
            k,
        ):

            mmr = (
                lambda_val *
                score
                -
                (
                    1.0 -
                    lambda_val
                )
                *
                max_sim_to_selected
            )

            mmr = mmr.masked_fill(
                selected_mask,
                float("-inf"),
            )

            probs = torch.softmax(
                mmr /
                temperature,
                dim=1,
            )

            new_idx = torch.multinomial(
                probs,
                num_samples=1,
            ).squeeze(
                1
            )

            selected[
                :,
                step
            ] = new_idx

            selected_mask.scatter_(
                1,
                new_idx[:, None],
                True,
            )

            sim_to_new = (
                cand_sim_matrix[
                    row,
                    :,
                    new_idx
                ]
            )

            max_sim_to_selected = torch.maximum(
                max_sim_to_selected,
                sim_to_new,
            )

        all_selected.append(
            selected.cpu().numpy()
        )

    return np.concatenate(
        all_selected,
        axis=0,
    ).astype(
        np.int64
    )


## 5. Metrics

In [ ]:

def future_distance(
    q_future,
    cand_future,
):
    return np.mean(
        (
            cand_future
            -
            q_future[
                :,
                None,
                :
            ]
        ) ** 2,
        axis=2,
    )


def gather2(
    x,
    idx,
):
    row = np.arange(
        len(x)
    )[:, None]

    return x[
        row,
        idx
    ]


def gather3(
    x,
    idx,
):
    row = np.arange(
        len(x)
    )[:, None]

    return x[
        row,
        idx,
        :
    ]


def metrics_from_selection(
    phase,
    local_selected,
    sigma=None,
):
    fdist = future_distance(
        phase[
            "q_future"
        ],
        phase[
            "cand_future"
        ],
    )

    selected_dist = gather2(
        fdist,
        local_selected,
    )

    analog = selected_dist.mean(
        axis=1
    )

    selected_future = gather3(
        phase[
            "cand_future"
        ],
        local_selected,
    )

    pred_uniform = selected_future.mean(
        axis=1
    )

    forecast_uniform = np.mean(
        (
            pred_uniform
            -
            phase[
                "q_future"
            ]
        ) ** 2,
        axis=1,
    )

    out = {
        "AnalogFutureMSE":
            analog.astype(
                np.float32
            ),

        "UniformForecastMSE":
            forecast_uniform.astype(
                np.float32
            ),
    }

    if sigma is not None:

        selected_sim = gather2(
            phase[
                "pattern_score"
            ],
            local_selected,
        )

        dist = (
            1.0 -
            selected_sim
        )

        weight = np.exp(
            -(
                dist ** 2
            )
            /
            (
                2.0 *
                sigma ** 2
            )
        )

        weight = (
            weight
            /
            (
                weight.sum(
                    axis=1,
                    keepdims=True,
                )
                +
                1e-12
            )
        )

        pred_weighted = np.sum(
            selected_future
            *
            weight[
                :,
                :,
                None
            ],
            axis=1,
        )

        forecast_weighted = np.mean(
            (
                pred_weighted
                -
                phase[
                    "q_future"
                ]
            ) ** 2,
            axis=1,
        )

        out[
            "SARAFWeightedForecastMSE"
        ] = forecast_weighted.astype(
            np.float32
        )

    return pd.DataFrame(
        out
    )


## 6. Run SARAF-MMR on the frozen test candidate pools

In [ ]:

SARAF_QUERY_RESULTS = {}
seed_rows = []
summary_rows = []

for dataset_name in DATASETS:

    params = SARAF_PARAMS[
        dataset_name
    ]

    for H in HORIZONS:

        phase = load_task(
            dataset_name,
            H,
        )

        pattern_sel = np.tile(
            np.arange(
                TOP_K,
                dtype=np.int64,
            )[
                None,
                :
            ],
            (
                len(
                    phase[
                        "q_idx"
                    ]
                ),
                1,
            ),
        )

        pattern_metrics = metrics_from_selection(
            phase,
            pattern_sel,
            sigma=None,
        )

        saraf_frames = []

        for seed in SARAF_SEEDS:

            sel = saraf_mmr_batched(
                phase[
                    "pattern_score"
                ],
                lambda_val=params[
                    "lambda"
                ],
                k=TOP_K,
                temperature=MMR_TEMPERATURE,
                seed=seed,
            )

            m = metrics_from_selection(
                phase,
                sel,
                sigma=params[
                    "sigma"
                ],
            )

            saraf_frames.append(
                m
            )

            seed_rows.append({
                "Dataset":
                    dataset_name,

                "Horizon":
                    H,

                "Seed":
                    seed,

                "Stationarity":
                    params[
                        "stationarity"
                    ],

                "Lambda":
                    params[
                        "lambda"
                    ],

                "Sigma":
                    params[
                        "sigma"
                    ],

                "AnalogFutureMSE":
                    float(
                        m[
                            "AnalogFutureMSE"
                        ].mean()
                    ),

                "UniformForecastMSE":
                    float(
                        m[
                            "UniformForecastMSE"
                        ].mean()
                    ),

                "SARAFWeightedForecastMSE":
                    float(
                        m[
                            "SARAFWeightedForecastMSE"
                        ].mean()
                    ),
            })

        saraf_mean = pd.DataFrame({
            col:
                np.stack(
                    [
                        x[
                            col
                        ].to_numpy()
                        for x in saraf_frames
                    ],
                    axis=0,
                ).mean(
                    axis=0
                )
            for col in saraf_frames[
                0
            ].columns
        })

        existing_path = (
            CONFIRM_DIR /
            f"query_level_{dataset_name}_H{H}.csv.gz"
        )

        assert existing_path.exists()

        existing = pd.read_csv(
            existing_path
        )

        assert len(existing) == len(
            pattern_metrics
        )

        q = pd.DataFrame({
            "Anchor":
                phase[
                    "q_anchor"
                ],

            "Pattern_AnalogFutureMSE":
                pattern_metrics[
                    "AnalogFutureMSE"
                ],

            "SARAF_AnalogFutureMSE":
                saraf_mean[
                    "AnalogFutureMSE"
                ],

            "Learned_AnalogFutureMSE":
                existing[
                    "Learned_AnalogFutureMSE"
                ].to_numpy(),

            "Pattern_UniformForecastMSE":
                pattern_metrics[
                    "UniformForecastMSE"
                ],

            "SARAF_UniformForecastMSE":
                saraf_mean[
                    "UniformForecastMSE"
                ],

            "SARAF_WeightedForecastMSE":
                saraf_mean[
                    "SARAFWeightedForecastMSE"
                ],

            "Learned_UniformForecastMSE":
                existing[
                    "Learned_RetrievalForecastMSE"
                ].to_numpy(),
        })

        SARAF_QUERY_RESULTS[
            (
                dataset_name,
                H
            )
        ] = q

        p_a = float(
            q[
                "Pattern_AnalogFutureMSE"
            ].mean()
        )

        s_a = float(
            q[
                "SARAF_AnalogFutureMSE"
            ].mean()
        )

        l_a = float(
            q[
                "Learned_AnalogFutureMSE"
            ].mean()
        )

        summary_rows.append({
            "Dataset":
                dataset_name,

            "Horizon":
                H,

            "Pattern":
                p_a,

            "SARAF_Matched":
                s_a,

            "Learned":
                l_a,

            "Pattern_to_SARAF_%":
                100.0 *
                (
                    p_a -
                    s_a
                )
                /
                p_a,

            "SARAF_to_Learned_%":
                100.0 *
                (
                    s_a -
                    l_a
                )
                /
                s_a,

            "SARAF_WeightedForecastMSE":
                float(
                    q[
                        "SARAF_WeightedForecastMSE"
                    ].mean()
                ),

            "Learned_UniformForecastMSE":
                float(
                    q[
                        "Learned_UniformForecastMSE"
                    ].mean()
                ),
        })

        q.to_csv(
            RESULT_DIR /
            f"query_level_{dataset_name}_H{H}.csv.gz",
            index=False,
            compression="gzip",
        )

seed_table = pd.DataFrame(
    seed_rows
)

summary_table = pd.DataFrame(
    summary_rows
)

display(
    summary_table
)

seed_table.to_csv(
    RESULT_DIR /
    "01_saraf_seed_results.csv",
    index=False,
)

summary_table.to_csv(
    RESULT_DIR /
    "02_saraf_matched_summary.csv",
    index=False,
)


## 7. Moving-block bootstrap against Pattern and Learned

In [ ]:

def moving_block_bootstrap(
    x,
    block_len,
    n_boot,
    seed,
):
    x = np.asarray(
        x,
        dtype=np.float64,
    )

    n = len(x)

    assert n >= block_len

    rng = np.random.default_rng(
        seed
    )

    n_blocks = math.ceil(
        n /
        block_len
    )

    max_start = (
        n -
        block_len
    )

    means = np.empty(
        n_boot,
        dtype=np.float64,
    )

    for b in range(n_boot):

        parts = []

        for _ in range(
            n_blocks
        ):

            start = int(
                rng.integers(
                    0,
                    max_start + 1,
                )
            )

            parts.append(
                x[
                    start:
                    start +
                    block_len
                ]
            )

        sample = np.concatenate(
            parts
        )[:n]

        means[b] = sample.mean()

    return {
        "ObservedImprovement":
            float(
                x.mean()
            ),

        "CI_2.5%":
            float(
                np.quantile(
                    means,
                    0.025,
                )
            ),

        "CI_97.5%":
            float(
                np.quantile(
                    means,
                    0.975,
                )
            ),

        "SignificantImprovement":
            bool(
                np.quantile(
                    means,
                    0.025,
                ) > 0
            ),
    }


bootstrap_rows = []

for dataset_name in DATASETS:

    for H in HORIZONS:

        q = SARAF_QUERY_RESULTS[
            (
                dataset_name,
                H
            )
        ]

        comparisons = [
            (
                "Pattern",
                "SARAF",
                q[
                    "Pattern_AnalogFutureMSE"
                ]
                -
                q[
                    "SARAF_AnalogFutureMSE"
                ],
            ),
            (
                "SARAF",
                "Learned",
                q[
                    "SARAF_AnalogFutureMSE"
                ]
                -
                q[
                    "Learned_AnalogFutureMSE"
                ],
            ),
            (
                "Pattern",
                "Learned",
                q[
                    "Pattern_AnalogFutureMSE"
                ]
                -
                q[
                    "Learned_AnalogFutureMSE"
                ],
            ),
        ]

        for baseline, proposed, diff in comparisons:

            tmp = pd.DataFrame({
                "Anchor":
                    q[
                        "Anchor"
                    ],

                "Diff":
                    diff,
            })

            anchor_diff = (
                tmp
                .groupby(
                    "Anchor"
                )[
                    "Diff"
                ]
                .mean()
                .sort_index()
                .to_numpy()
            )

            r = moving_block_bootstrap(
                anchor_diff,
                BLOCK_ANCHORS,
                N_BOOT,
                seed=(
                    9000
                    +
                    H
                    +
                    len(
                        bootstrap_rows
                    )
                ),
            )

            r.update({
                "Dataset":
                    dataset_name,

                "Horizon":
                    H,

                "Baseline":
                    baseline,

                "Proposed":
                    proposed,
            })

            bootstrap_rows.append(
                r
            )

bootstrap_table = pd.DataFrame(
    bootstrap_rows
)

display(
    bootstrap_table
)

bootstrap_table.to_csv(
    RESULT_DIR /
    "03_saraf_matched_bootstrap.csv",
    index=False,
)


## 8. Paper-ready external-baseline table

In [ ]:

paper_rows = []

for dataset_name in DATASETS:

    x = summary_table[
        summary_table[
            "Dataset"
        ] ==
        dataset_name
    ]

    paper_rows.append({
        "Dataset":
            dataset_name,

        "Pattern":
            float(
                x[
                    "Pattern"
                ].mean()
            ),

        "SARAF-Matched":
            float(
                x[
                    "SARAF_Matched"
                ].mean()
            ),

        "Learned":
            float(
                x[
                    "Learned"
                ].mean()
            ),

        "Pattern->SARAF_%":
            float(
                x[
                    "Pattern_to_SARAF_%"
                ].mean()
            ),

        "SARAF->Learned_%":
            float(
                x[
                    "SARAF_to_Learned_%"
                ].mean()
            ),

        "LearnedBeatsSARAF_Horizons":
            int(
                (
                    x[
                        "Learned"
                    ]
                    <
                    x[
                        "SARAF_Matched"
                    ]
                ).sum()
            ),
    })

paper_table = pd.DataFrame(
    paper_rows
)

display(
    paper_table
)

paper_table.to_csv(
    RESULT_DIR /
    "04_paper_external_baseline_table.csv",
    index=False,
)


## 9. Interpretation guardrails

In [ ]:

n_tasks = len(
    summary_table
)

learned_beats_saraf = int(
    (
        summary_table[
            "Learned"
        ]
        <
        summary_table[
            "SARAF_Matched"
        ]
    ).sum()
)

saraf_beats_pattern = int(
    (
        summary_table[
            "SARAF_Matched"
        ]
        <
        summary_table[
            "Pattern"
        ]
    ).sum()
)

decision = pd.DataFrame(
    [
        {
            "Tasks":
                n_tasks,

            "SARAFBeatsPattern":
                saraf_beats_pattern,

            "LearnedBeatsSARAF":
                learned_beats_saraf,

            "RecommendedPaperLabel":
                "SARAF retrieval rule (matched protocol)",

            "ImportantCaveat":
                (
                    "This is a matched-protocol evaluation of the "
                    "public SARAF retrieval rule, not the authors' "
                    "native seq_len=720 forecasting reproduction."
                ),
        }
    ]
)

display(
    decision
)

decision.to_csv(
    RESULT_DIR /
    "05_external_baseline_decision.csv",
    index=False,
)



# How to use the result in the paper

If Learned is stronger than SARAF-Matched, add a compact comparison such as:

| Dataset | Pattern | SARAF retrieval rule | Ours |
|---|---:|---:|---:|

and state that under the same candidate pool and evaluation protocol, future-supervised relevance learning is competitive with or improves over a recent stationarity-aware retrieval rule.

If SARAF-Matched is stronger on some datasets, this is still informative. For example, stronger SARAF behavior on Exchange would be consistent with Exchange being candidate-global / nonstationarity-dominated, whereas stronger Learned behavior on Solar or Traffic would support query-specific future supervision.

The objective of this last experiment is **external validation**, not forcing a win on every dataset.


## 10. Optional exact native SARAF reproduction

In [ ]:

RUN_OFFICIAL_NATIVE = False

SARAF_REPO = Path(os.environ.get(
    "SARAF_REPO", REPO_ROOT / "external" / "SARAF"
)).expanduser().resolve()

if RUN_OFFICIAL_NATIVE:

    import subprocess

    if not SARAF_REPO.exists():

        subprocess.run(
            [
                "git",
                "clone",
                "https://github.com/"
                "ShiqiaoZhou/SARAF.git",
                str(
                    SARAF_REPO
                ),
            ],
            check=True,
        )

    print(
        "Official SARAF repository:",
        SARAF_REPO
    )

    print(
        "Recommended native scripts:"
    )

    print(
        "bash scripts/ETTh_720.sh"
    )

    print(
        "bash scripts/elec_720.sh"
    )

    print(
        "bash scripts/exchange_rate_720.sh"
    )

    print(
        "bash scripts/traffic_720.sh"
    )

    print(
        "bash scripts/solar_720.sh"
    )

    print(
        "\nNative results are appended to "
        "result_long_term_forecast.txt by the public code."
    )



# Final stopping rule

After this external-baseline experiment:

\[
oxed{	ext{Do not tune the proposed retriever further.}}
\]

The remaining work is:

1. incorporate the external baseline result;
2. finalize the main quantitative tables;
3. refine the ICLR text and figures;
4. complete reproducibility details.
